In [2]:
! pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    python-dotenv

In [2]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

/opt/anaconda3/envs/jub/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/9f/00d3gq1s0w3_0m391myx76400000gn/T/ipykernel_1679/1190506230.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [4]:
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

# --- YOUR CODE HERE ---

In [3]:
# Load the variables from the .env file into the environment
load_dotenv()

#2 Access your key securely 
api_key = os.getenv('GROQ_API_KEY')

In [57]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful "
            "and multi-agent AI applications."
        ),
        metadata={"source": "langgraph_notes"},
    ),
    Document(
        page_content=(
            "Inception BD is a Edtech Platform "
            "and provides courses on AI."
        ),
        metadata={"source": "inception_notes"},
    ),
    Document(
        page_content=(
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."
        ),
        metadata={"source": "rag_notes"},
    ),
    Document(
        page_content=(
            "Groq provides fast inference for supported large "
            "language models through the Groq API."
        ),
        metadata={"source": "groq_notes"},
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity-search library. "
            "It can retrieve documents whose embeddings are close "
            "to the query embedding."
        ),
        metadata={"source": "faiss_notes"},
    ),
    Document(
        page_content=(
            "Jubayer is AI Enginner "
            "He is working"
        ),
        metadata={'source':"Jubayer"}
    )
]


In [29]:
documents

[Document(metadata={'source': 'langgraph_notes'}, page_content='LangGraph is a framework for building stateful and multi-agent AI applications.'),
 Document(metadata={'source': 'inception_notes'}, page_content='Inception BD is a Edtech Platform and provides courses on AI.'),
 Document(metadata={'source': 'rag_notes'}, page_content='RAG stands for Retrieval-Augmented Generation. It retrieves relevant information before generating an answer.'),
 Document(metadata={'source': 'groq_notes'}, page_content='Groq provides fast inference for supported large language models through the Groq API.'),
 Document(metadata={'source': 'faiss_notes'}, page_content='FAISS is a vector similarity-search library. It can retrieve documents whose embeddings are close to the query embedding.')]

In [5]:
type(documents[0])

langchain_core.documents.base.Document

### embeding mode all minillmv-6

In [58]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs = {
        "normalize_embeddings": True, 
    },
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7938.21it/s]


### Create the FAISS Vector strore and stroe embedings 


In [60]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

save to the local 
load local

In [61]:
vector_store.save_local('1faiss_index')

In [62]:
vector_store = FAISS.load_local('1faiss_index',embeddings, allow_dangerous_deserialization=True)

### Crate Retriver 
vetor is knwolaged bas 

In [63]:
retriver = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {"k": 2}# k = 2 two result present or k=3 result present
)

### LLM mode l connection


In [17]:
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

In [ ]:
llm = ChatGroq(
    model= 'llama-3.1-8b-instant',
    temperature=0, # not be creative 0, use to be knowledge  1 is crativeat 
    max_retries=2, # 2 try to connect two times
    api_key= 'api'
)

In [40]:
response =llm.invoke("Hi tell  me about you google ")

In [41]:
response.content

'I\'m not Google, but I\'m a large language model (LLM) provided by Google. I\'m an AI designed to assist and communicate with users in a helpful and informative way.\n\nHere\'s a brief overview of Google:\n\n**What is Google?**\n\nGoogle is a multinational technology company that specializes in Internet-related services and products. It was founded in 1998 by Larry Page and Sergey Brin while they were Ph.D. students at Stanford University in California.\n\n**Google\'s Products and Services**\n\nGoogle offers a wide range of products and services, including:\n\n1. **Search Engine**: Google\'s search engine is the most widely used search engine in the world, allowing users to search for information on the web.\n2. **Google Ads**: A platform for businesses to create and display ads on Google\'s search engine and other websites.\n3. **Google Maps**: A mapping service that provides directions, satellite imagery, and street views of locations around the world.\n4. **YouTube**: A video-shari

In [23]:
response =llm.invoke("tell me about uk")

In [24]:
response.content

'The United Kingdom (UK) is a sovereign state located in Northwest Europe, comprising four constituent countries: England, Scotland, Wales, and Northern Ireland. Here\'s an overview of the UK:\n\n**Geography and Climate**\n\nThe UK is an archipelago of over 6,000 islands, with the largest being Great Britain (England, Scotland, and Wales) and Ireland (Northern Ireland). The UK shares a border with the Republic of Ireland to the west. The country\'s terrain is diverse, with mountains, hills, and coastal plains. The climate is generally temperate, with mild winters and cool summers.\n\n**History**\n\nThe UK has a rich and complex history, with evidence of human habitation dating back to the Mesolithic era. The Romans conquered the island in 43 AD, followed by the Anglo-Saxons, Vikings, and Normans. The UK became a unified state in 1707, with the Acts of Union merging England and Scotland. The Industrial Revolution transformed the country\'s economy, and the UK became a major world power.

#### Create the prompt

In [64]:
prompt = ChatPromptTemplate.from_template(
            """
            You are a helpful assistant.

            Answer the question using only the provided context.

            If the answer is not present in the context, say:
            "I do not know based on the provided context."

            Context:
            {context}

            Question:
            {question}

            Answer:
            """
)

### Format retrieved document

In [65]:
def format_documents(retrieved_documents: list[Document]) -> str:
    return "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )

## Build the RAG chain

In [67]:
rag_chain =(
    {
        "context": retriver | format_documents,
        'question': RunnablePassthrough(),
    }
    |prompt
    |llm
    |StrOutputParser() # make nice out put 

)


### Ask question


In [68]:
def ask_question(question: str) ->str:
    if not question.strip():
        raise ValueError('Question cannot be empty. ')

    return rag_chain.invoke(question)

In [69]:
if __name__ =="__main__":
    while True:
        user_question =input(
            "\nAsk a question of type 'text' : "
        ).strip()

        if user_question.lower() == 'exit':
            print("Application closed ")
            break

        try:
            answer = ask_question(user_question)

            print("\nAnswer :")
            print(answer)
        except Exception as error:
            print(f"\nError : {error}")


Answer :
Jubayer is AI Engineer.

Answer :
RAG stands for Retrieval-Augmented Generation. It retrieves relevant information before generating an answer.

Answer :
Inception BD is a Edtech Platform and provides courses on AI.
Application closed 
